# Design and Test MCP Tools

## Find nearest Store

### Postal Code

In [15]:
from pymongo import MongoClient
from dotenv import dotenv_values

env_path = "/workspace/.env"
config = dotenv_values(env_path)

In [16]:
client = MongoClient(config['MONGO_CONNECTION_STRING'])
db = client["stylistai"]
stores_collection = db["stores"]

stores_collection.find_one({'country': 'Portugal'})

{'_id': ObjectId('6969264a1d2d99038365937e'),
 'store_id': 'PT0301',
 'Fri_open_hours': '10:00-21:00',
 'Mon_open_hours': '10:00-21:00',
 'Sat_open_hours': '10:00-21:00',
 'Sun_open_hours': '10:00-20:00',
 'Thu_open_hours': '10:00-21:00',
 'Tue_open_hours': '10:00-21:00',
 'Unnamed: 0': 3132,
 'Wed_open_hours': '10:00-21:00',
 'address_string': 'Rua do Carmo, 42, 5º andar;;1249-270;Lisboa;Lisboa',
 'city': 'Lisbon',
 'country': 'Portugal',
 'countryCode': 'PT',
 'latitude': 38.71166293870208,
 'longitude': -9.13964795699258,
 'name': 'Rua do Carmo, 42, 5º andar',
 'phone': '+351-800780330',
 'postal_code': '1100-062',
 'state': 'Lisboa',
 'storeClass': 'Flagship',
 'storeCode': 'PT0301',
 'streetName1': 'Rua do Carmo, 42, 5º andar',
 'streetName2': None,
 'timeZoneIndex': 85.0}

In [17]:
portugal_stores = stores_collection.find({'country': 'Portugal'})
len(list(portugal_stores))

28

In [18]:
docs = stores_collection.find({"$and": [{'country': 'Portugal'}, {'postal_code': '1100-062'}]})
list(docs)

[{'_id': ObjectId('6969264a1d2d99038365937e'),
  'store_id': 'PT0301',
  'Fri_open_hours': '10:00-21:00',
  'Mon_open_hours': '10:00-21:00',
  'Sat_open_hours': '10:00-21:00',
  'Sun_open_hours': '10:00-20:00',
  'Thu_open_hours': '10:00-21:00',
  'Tue_open_hours': '10:00-21:00',
  'Unnamed: 0': 3132,
  'Wed_open_hours': '10:00-21:00',
  'address_string': 'Rua do Carmo, 42, 5º andar;;1249-270;Lisboa;Lisboa',
  'city': 'Lisbon',
  'country': 'Portugal',
  'countryCode': 'PT',
  'latitude': 38.71166293870208,
  'longitude': -9.13964795699258,
  'name': 'Rua do Carmo, 42, 5º andar',
  'phone': '+351-800780330',
  'postal_code': '1100-062',
  'state': 'Lisboa',
  'storeClass': 'Flagship',
  'storeCode': 'PT0301',
  'streetName1': 'Rua do Carmo, 42, 5º andar',
  'streetName2': None,
  'timeZoneIndex': 85.0}]

In [19]:
def get_near_stores(city: str, db: str = "stylistai", collection: str = "stores") -> dict:
    client = MongoClient(config['MONGO_CONNECTION_STRING'])
    db = client[db]
    stores_collection = db[collection]
    docs = stores_collection.find({"$and": [{'country': 'Portugal'}, {'city': city}]})
    return list(docs)

In [20]:
get_near_stores(city='Lisbon')

[{'_id': ObjectId('6969264a1d2d99038365937e'),
  'store_id': 'PT0301',
  'Fri_open_hours': '10:00-21:00',
  'Mon_open_hours': '10:00-21:00',
  'Sat_open_hours': '10:00-21:00',
  'Sun_open_hours': '10:00-20:00',
  'Thu_open_hours': '10:00-21:00',
  'Tue_open_hours': '10:00-21:00',
  'Unnamed: 0': 3132,
  'Wed_open_hours': '10:00-21:00',
  'address_string': 'Rua do Carmo, 42, 5º andar;;1249-270;Lisboa;Lisboa',
  'city': 'Lisbon',
  'country': 'Portugal',
  'countryCode': 'PT',
  'latitude': 38.71166293870208,
  'longitude': -9.13964795699258,
  'name': 'Rua do Carmo, 42, 5º andar',
  'phone': '+351-800780330',
  'postal_code': '1100-062',
  'state': 'Lisboa',
  'storeClass': 'Flagship',
  'storeCode': 'PT0301',
  'streetName1': 'Rua do Carmo, 42, 5º andar',
  'streetName2': None,
  'timeZoneIndex': 85.0},
 {'_id': ObjectId('6969264a1d2d990383659380'),
  'store_id': 'PT0303',
  'Fri_open_hours': '10:00-23:00',
  'Mon_open_hours': '10:00-23:00',
  'Sat_open_hours': '10:00-23:00',
  'Sun_op

### Address

In [21]:
import googlemaps
import time
from googlemaps.exceptions import TransportError, Timeout, ApiError

GOOGLE_MAPS_API = config['GOOGLE_MAPS_API']

gmaps = googlemaps.Client(key=GOOGLE_MAPS_API)

def get_customer_coordinates(address: str, max_retries=5):
    """
    Query Google Maps API to get latitude and longitude from a given address.
    """
    for attempt in range(max_retries):
        try:
            time.sleep(0.1)
            geocode_result = gmaps.geocode(address)
            if geocode_result:
                return geocode_result[0]['geometry']['location']
            else:
                return None

        except (TransportError, Timeout) as e:
            time.sleep(2 ** attempt)

        except ApiError as e:
            print(f"API Error: {e}")
            return None
        
    return None


In [22]:
# store_coords = get_customer_coordinates(address='R. do Carmo 42, 1100-062 Lisboa, Portugal')
# print(store_coords)

In [23]:
# reverse_geocode_result = gmaps.reverse_geocode((38.71166293870208, -9.13964795699258))
# print(reverse_geocode_result)

In [24]:
# lisbon_address = get_customer_coordinates("Largo da Sé 1, 1100-585 Lisboa, Portugal")

In [25]:
from geopy.geocoders import Nominatim
geolocator = Nominatim(user_agent="StylistAI")
location = geolocator.geocode("Largo da Sé 1, 1100-585 Lisboa, Portugal")
print(location.address)
print((location.latitude, location.longitude))

Largo da Sé, Alfama, Sé, Santa Maria Maior, Madalena, Lisboa, 1100-501, Portugal
(38.7098999, -9.1333811)


In [ ]:
from functools import lru_cache
from typing import Optional, Tuple
from geopy.geocoders import Nominatim
from geopy.extra.rate_limiter import RateLimiter
from geopy.exc import GeocoderTimedOut, GeocoderUnavailable, GeocoderServiceError

_geolocator = Nominatim(user_agent="StylistAI")
_geocode = RateLimiter(
    _geolocator.geocode,
    min_delay_seconds=1.0,
    max_retries=3,
    error_wait_seconds=2.0,
    swallow_exceptions=False
)

def _normalize_address(address: str) -> str:
    return " ".join(address.strip().split())

@lru_cache(maxsize=20_000)
def get_customer_coordinates(address: str) -> Optional[Tuple[float, float]]:
    """
    Geocode an address using Nominatim (OpenStreetMap).
    Returns (lat, lon) or None.

    Cached (LRU) to avoid repeated calls in agent flows.
    Rate-limited to respect Nominatim usage.
    """
    coordinates = {}

    addr = _normalize_address(address)
    if not addr:
        return None

    try:
        location = _geocode(addr, exactly_one=True)
        if location is None:
            return None
        coordinates['lat'] = location.latitude
        coordinates['lng'] = location.longitude
        return coordinates

    except (GeocoderTimedOut, GeocoderUnavailable, GeocoderServiceError):
        return None


In [27]:
coordinates = get_customer_coordinates("Largo da Sé 1, 1100-585 Lisboa, Portugal")
coordinates

{'lat': 38.7098999, 'lng': -9.1333811}

In [29]:
import geopy.distance

# coords_1 = (store_coords['lat'], store_coords['lng'])
# coords_2 = (lisbon_address['lat'], lisbon_address['lng'])

# print(geopy.distance.geodesic(coords_1, coords_2).km)


In [30]:
def get_closest_store(address: str, city: str) -> str:
    distances = {}

    stores = get_near_stores(city=city)
    customer_coordinates = get_customer_coordinates(address=address)
    customer_coords = (customer_coordinates['lat'], customer_coordinates['lng'])

    for doc in stores:
        store_coords = (doc['latitude'], doc['longitude'])
        distance = geopy.distance.geodesic(customer_coords, store_coords).km
        distances[str(doc['_id'])] = distance

    return distances


In [31]:
distances = get_closest_store(address="Largo da Sé 1, 1100-585 Lisboa, Portugal", city='Lisbon')
distances

{'6969264a1d2d99038365937e': 0.5791500586108235,
 '6969264a1d2d990383659380': 23.35651312707066,
 '6969264a1d2d990383659383': 13.99222913058461,
 '6969264a1d2d990383659385': 6.903714493379061,
 '6969264a1d2d990383659386': 7.179864931823647,
 '6969264a1d2d99038365938b': 10.514774347141328}

In [32]:
min_pair = min(distances.items(), key=lambda x: x[1])
min_pair

('6969264a1d2d99038365937e', 0.5791500586108235)

In [33]:
from bson.objectid import ObjectId

def get_store_info(store_id: str, db: str = "stylistai", collection: str = "stores") -> dict:
    client = MongoClient(config['MONGO_CONNECTION_STRING'])
    db = client[db]
    stores_collection = db[collection]

    try:
        query_id = ObjectId(store_id)
    except Exception:
        return None
    
    return stores_collection.find_one({'_id': query_id})

get_store_info('6969264a1d2d99038365937e')

{'_id': ObjectId('6969264a1d2d99038365937e'),
 'store_id': 'PT0301',
 'Fri_open_hours': '10:00-21:00',
 'Mon_open_hours': '10:00-21:00',
 'Sat_open_hours': '10:00-21:00',
 'Sun_open_hours': '10:00-20:00',
 'Thu_open_hours': '10:00-21:00',
 'Tue_open_hours': '10:00-21:00',
 'Unnamed: 0': 3132,
 'Wed_open_hours': '10:00-21:00',
 'address_string': 'Rua do Carmo, 42, 5º andar;;1249-270;Lisboa;Lisboa',
 'city': 'Lisbon',
 'country': 'Portugal',
 'countryCode': 'PT',
 'latitude': 38.71166293870208,
 'longitude': -9.13964795699258,
 'name': 'Rua do Carmo, 42, 5º andar',
 'phone': '+351-800780330',
 'postal_code': '1100-062',
 'state': 'Lisboa',
 'storeClass': 'Flagship',
 'storeCode': 'PT0301',
 'streetName1': 'Rua do Carmo, 42, 5º andar',
 'streetName2': None,
 'timeZoneIndex': 85.0}

In [34]:
def get_country_cities(country: str = "Portugal",
                    db: str = "stylistai",
                    collection: str = "stores"):
    
    client = MongoClient(config['MONGO_CONNECTION_STRING'])
    db = client[db]
    stores_collection = db[collection]
    results = stores_collection.find({'country': country})
    docs = list(results)
    cities = [doc['city'] for doc in docs]
    return set(cities)